# Laboratorio 26 — Qiskit Runtime: simulador local y conexión a IBM Quantum

Este laboratorio muestra cómo usar **Qiskit Runtime** con primitivas V2:
- `StatevectorEstimator` / `StatevectorSampler` para simulación local (sin cuenta IBM).
- `QiskitRuntimeService` para conectar con hardware real de IBM Quantum (requiere cuenta).

**Prerrequisitos:** Módulos 04 (Qiskit runtime y primitives), 11 (VQE/QAOA).

**Nota:** Las celdas marcadas con `[IBM Cloud]` requieren credenciales IBM Quantum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import EfficientSU2
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

print('Qiskit Runtime primitives importadas correctamente')

## 1. Primitivas V2 — Estimator local

El `StatevectorEstimator` calcula valores esperados ⟨ψ|H|ψ⟩ exactamente.
Es el estándar para simulación local antes de enviar a hardware real.

In [ ]:
# Hamiltoniano H2 simplificado
H = SparsePauliOp.from_list([
    ('II', -1.0523), ('ZI', 0.3979), ('IZ', -0.3979),
    ('ZZ', -0.0112), ('XX',  0.1809),
])

ansatz = EfficientSU2(2, reps=2, entanglement='linear')
estimator = StatevectorEstimator()

# Una sola evaluación
np.random.seed(42)
params = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
bound = ansatz.assign_parameters(params)

result = estimator.run([(bound, H)]).result()
energia = float(result[0].data.evs)
print(f'⟨H⟩ = {energia:.6f} Ha')

# Batch: múltiples configuraciones de parámetros
params_batch = [np.random.uniform(-np.pi, np.pi, ansatz.num_parameters) for _ in range(5)]
pubs = [(ansatz.assign_parameters(p), H) for p in params_batch]
resultados_batch = estimator.run(pubs).result()
energias_batch = [float(r.data.evs) for r in resultados_batch]
print(f'Batch de 5 configuraciones:')
for i, e in enumerate(energias_batch):
    print(f'  Config {i+1}: {e:.6f} Ha')

## 2. Primitivas V2 — Sampler local

El `StatevectorSampler` simula mediciones y devuelve distribuciones de probabilidad.

In [ ]:
sampler = StatevectorSampler()

# Circuito de Bell
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure_all()

job = sampler.run([qc_bell], shots=4096)
result = job.result()
counts = result[0].data.meas.get_counts()
print('Distribución del estado de Bell |Φ+⟩:')
for estado, n in sorted(counts.items()):
    print(f'  |{estado}⟩: {n} ({100*n/4096:.1f}%)')

# Visualizar
from qiskit.visualization import plot_histogram
fig = plot_histogram(counts)
plt.title('Estado de Bell — distribución de medición')
plt.show()

## 3. VQE completo con primitivas V2

In [ ]:
from scipy.optimize import minimize

estimator = StatevectorEstimator()
historia = []

def funcion_coste(params):
    bound = ansatz.assign_parameters(params)
    res = estimator.run([(bound, H)]).result()
    e = float(res[0].data.evs)
    historia.append(e)
    return e

np.random.seed(0)
x0 = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
resultado = minimize(funcion_coste, x0, method='COBYLA', options={'maxiter': 300})

print(f'VQE completado:')
print(f'  Energía mínima: {resultado.fun:.6f} Ha')
print(f'  Referencia FCI: -1.8572 Ha')
print(f'  Error: {abs(resultado.fun - (-1.8572))*1000:.2f} mHa')
print(f'  Iteraciones: {len(historia)}')

plt.figure(figsize=(9, 4))
plt.plot(historia, color='#3498db', linewidth=1.5)
plt.axhline(y=-1.8572, color='k', linestyle=':', alpha=0.7, label='FCI ref. −1.8572 Ha')
plt.xlabel('Iteración'); plt.ylabel('⟨H⟩ (Hartree)')
plt.title('Convergencia VQE con StatevectorEstimator (Qiskit Runtime V2)')
plt.legend(); plt.grid(alpha=0.2)
plt.show()

## 4. [IBM Cloud] Conexión a hardware real

Esta sección requiere una cuenta en [IBM Quantum](https://quantum.ibm.com/).
Obtén tu API token en: `quantum.ibm.com → Account → API Token`.

> Las celdas siguientes están comentadas para no ejecutarse en CI.

In [ ]:
# [IBM Cloud] — descomentar para usar con credenciales reales
#
# from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2, SamplerV2
#
# Guardar credenciales (solo una vez, se almacenan en ~/.qiskit/qiskit-ibm.json):
# QiskitRuntimeService.save_account(channel='ibm_quantum', token='TU_TOKEN_AQUI')
#
# Cargar servicio y listar backends disponibles:
# service = QiskitRuntimeService(channel='ibm_quantum')
# backends = service.backends(operational=True, simulator=False)
# for b in backends:
#     print(f'{b.name}: {b.num_qubits} qubits, QV={b.configuration().quantum_volume}')

print('[IBM Cloud] Celda comentada — requiere token de IBM Quantum para ejecutar')

In [ ]:
# [IBM Cloud] — Ejecutar VQE en hardware real
#
# backend = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
# print(f'Backend seleccionado: {backend.name}')
#
# # Transpilar para el backend específico
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
# ansatz_transpilado = pm.run(ansatz)
#
# # Crear sesión (agrupa trabajos para reducir latencia entre trabajos)
# from qiskit_ibm_runtime import Session
# with Session(backend=backend) as session:
#     estimator_real = EstimatorV2(session=session)
#     # Evaluar una configuración
#     bound_t = ansatz_transpilado.assign_parameters(resultado.x)
#     H_transpilado = H.apply_layout(ansatz_transpilado.layout)
#     res_real = estimator_real.run([(bound_t, H_transpilado)]).result()
#     print(f'Energia en hardware real: {float(res_real[0].data.evs):.4f} Ha')

print('[IBM Cloud] Celda comentada — requiere acceso a IBM Quantum para ejecutar')

## 5. Comparativa: simulador local vs. hardware real

Con hardware real se añade ruido de puerta, readout y decoherencia.
La comparativa es fundamental para entender las limitaciones de NISQ.

| Aspecto | StatevectorEstimator (local) | Hardware IBM (real) |
|---|---|---|
| Velocidad | Muy rápido (< 1s) | Lento (cola + ejecución: minutos) |
| Precisión | Exacta (sin ruido) | Degradada por ruido físico |
| Qubits | Ilimitados (limitado por RAM) | 127-1121 qubits físicos |
| Coste | Gratuito | Créditos IBM Quantum |
| Uso recomendado | Desarrollo y debugging | Validación en hardware real |

**Flujo recomendado:**
1. Desarrollar y optimizar con simulador local.
2. Validar con `AerSimulator(noise_model=...)` para modelar el ruido esperado.
3. Ejecutar en hardware real con el menor número de shots necesarios.
4. Comparar resultados con mitigación de errores (ZNE, readout correction).

In [ ]:
# Simulación de hardware real con AerSimulator y modelo de ruido
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
from qiskit.primitives import StatevectorEstimator

# Modelo de ruido representativo de IBM Heron (2024)
# T1 = 300 μs, T2 = 200 μs, F_1Q = 99.95%, F_2Q = 99.7%
noise_model_ibm = NoiseModel()
noise_model_ibm.add_all_qubit_quantum_error(depolarizing_error(0.0005, 1), ['u', 'h', 'rz'])
noise_model_ibm.add_all_qubit_quantum_error(depolarizing_error(0.003, 2), ['cx'])

# Comparativa: ideal vs. ruidoso para VQE
E_ideal = resultado.fun

sim_ruidoso = AerSimulator(noise_model=noise_model_ibm)
qc_meas = bound.copy()
qc_meas.measure_all()

job = sim_ruidoso.run(qc_meas, shots=8192)
counts_ruidoso = job.result().get_counts()

print('Comparativa ideal vs. simulador con ruido IBM Heron (2024):')
print(f'  Energía VQE ideal:   {E_ideal:.6f} Ha')
print(f'  Distribución ruidosa (top 4 estados):')
top4 = sorted(counts_ruidoso.items(), key=lambda x: x[1], reverse=True)[:4]
for estado, n in top4:
    print(f'    |{estado}⟩: {n} ({100*n/8192:.1f}%)')
print(f'  Para energía exacta con ruido: usar ZNE (ver laboratorio 24)')